This notebook is for training, following the guide from [Unitree RL Mjlab](https://github.com/unitreerobotics/unitree_rl_mjlab) project. 

# Setup and Install

In [ ]:
!sudo apt-get update

In [ ]:
!sudo apt install -y libyaml-cpp-dev libboost-all-dev libeigen3-dev libspdlog-dev libfmt-dev

In [ ]:
!git clone https://github.com/unitreerobotics/unitree_rl_mjlab.git

In [ ]:
!python3 -m pip install --upgrade pip

In [ ]:
%cd unitree_rl_mjlab
%pip install -e .

In [ ]:
# Should be the unitree_rl_mjlab directory
%pwd

# Training

Insert your own [wandb](https://wandb.ai/site/) API key for logging. 

In [ ]:
import wandb

# Add your W&B API key here
wandb.login(key='')

Run the training script. Some important parameters are:
- `--gpu-ids='[0,1]'`: The GPUs used for training. (default: GPU 0)
- `--env.scene.num-envs=4096`: Number of parallel environments. (default: 1)
- `--agent.max-iterations=1001`: The maximum number of iterations. (default: 10001)

To see the full list of parameters: 
```
!python scripts/train.py Unitree-G1-Flat --help
```

In [ ]:
!python scripts/train.py Unitree-G1-Flat \
  --gpu-ids='[0,1]' \
  --env.scene.num-envs=4096 --agent.max-iterations=1001

# Deploying

Run the code to visualize policy behavior in MuJoCo. This part should be done locally as you can't access localhost on cloud-based platform like Google Colab or Kaggle. If you trained using cloud platforms, download the model file and run the code locally.  

In [10]:
# !python scripts/play.py Unitree-G1-Flat --checkpoint_file="logs/rsl_rl/g1_velocity/2026-03-19_20-20-48/model_900.pt"

To run simulation deployment, first ensure you have cmake

In [ ]:
!sudo apt update
!sudo apt install cmake -y

In [ ]:
%cd /kaggle/working/unitree_rl_mjlab/simulate
!git clone https://github.com/unitreerobotics/unitree_sdk2.git
%cd unitree_sdk2
%mkdir build
%cd build
!cmake .. -DCMAKE_INSTALL_PREFIX=/opt/unitree_robotics
!make -j
!sudo make install

In [ ]:
!apt-get update
!apt-get install -y libglfw3-dev

In [ ]:
%cd /kaggle/working/unitree_rl_mjlab/simulate
%mkdir build
%cd build

In [ ]:
!cmake .. && make -j8

In [ ]:
!/kaggle/working/unitree_rl_mjlab/simulate/build/unitree_mujoco

In [ ]:
%cd deploy/robots/g1/build
!./g1_ctrl --network=lo

# Reverse Implementing 
From this section, I will attempt to re-implement the training process. With mjlab and rsl_rl libraries, the simple training pipeline is relatively simple.  

In [ ]:
import mjlab.tasks 
import torch
from mjlab.tasks.registry import list_tasks, load_env_cfg, load_rl_cfg

from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import RslRlVecEnvWrapper

from rsl_rl.runners import OnPolicyRunner
from dataclasses import asdict

task_id = "Mjlab-Velocity-Rough-Unitree-G1"  # Example task
env_cfg = load_env_cfg(task_id)
agent_cfg = load_rl_cfg(task_id)

env_cfg.scene.num_envs = 1024

device = "cuda:0" if torch.cuda.is_available() else "cpu"
env = ManagerBasedRlEnv(cfg=env_cfg, device=device)
env = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

train_cfg = asdict(agent_cfg)

for key in ("actor", "critic"):
      if key in train_cfg:
        for opt in ("cnn_cfg", "distribution_cfg"):
          if train_cfg[key].get(opt) is None:
            train_cfg[key].pop(opt, None)

runner = OnPolicyRunner(env, train_cfg, "logs/notebook_test", device)
runner.learn(num_learning_iterations=1000, init_at_random_ep_len=True)